## Verteilte Klassifikation mit MLlib

Dieses Beispiel zeigt die Verwendung der Spark Bibliothek MLlib für die Erkennung von Sarkasmus.

In [9]:
import os

from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml import Pipeline
from pyspark.ml.feature import (RegexTokenizer, StopWordsRemover, NGram,
                                CountVectorizer, IDF, VectorAssembler)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql import functions as F
import numpy as np

spark = SparkSession.builder.appName("MLLibClassification").getOrCreate()


Spark-Version: 4.0.2


### Datensatz laden

In diesem Schritt wird ein Beispiel-Datensatz für Sarkasmus-Erkennung geladen. Die Datei enthält Nachrichtenüberschriften (`headline`), ein Label (`is_sarcastic`), das angibt, ob die Überschrift sarkastisch ist, sowie einen Link zum ursprünglichen Artikel (`article_link`).

Zunächst wird der Datensatz mit `wget` heruntergeladen und anschließend mit Spark als JSON-Datei eingelesen. Mit `printSchema()` wird die Struktur der Daten angezeigt, und `show()` gibt einige Beispielzeilen aus, um einen ersten Eindruck vom Datensatz zu erhalten.

In [3]:
# Lade die v2-JSON (Headline, is_sarcastic, article_link)
!wget -q https://raw.githubusercontent.com/iTerner/Sarcasm-Detection/master/Sarcasm_Headlines_Dataset_v2.json -O sarcasm_v2.json


raw = spark.read.json("sarcasm_v2.json")
raw.printSchema()
raw.select("headline","is_sarcastic").show(3, truncate=80)

root
 |-- article_link: string (nullable = true)
 |-- headline: string (nullable = true)
 |-- is_sarcastic: long (nullable = true)

+-------------------------------------------------------------------------------+------------+
|                                                                       headline|is_sarcastic|
+-------------------------------------------------------------------------------+------------+
|                  thirtysomething scientists unveil doomsday clock of hair loss|           1|
|dem rep. totally nails why congress is falling short on gender, racial equality|           0|
|                              eat your veggies: 9 deliciously different recipes|           0|
+-------------------------------------------------------------------------------+------------+
only showing top 3 rows


### Daten vorbereiten und Trainings-/Testdaten erzeugen

In diesem Schritt werden nur die für das Modell relevanten Spalten ausgewählt. Die Überschrift (`headline`) wird in `text` umbenannt, und das Sarkasmus-Label (`is_sarcastic`) wird in einen Integer-Wert (`label`) umgewandelt. Anschließend werden Zeilen mit fehlenden Werten entfernt.

Der bereinigte Datensatz wird danach zufällig in zwei Teile aufgeteilt:  
**80 % Trainingsdaten** und **20 % Testdaten**.  

Mit `count()` wird abschließend überprüft, wie viele Datensätze insgesamt sowie in den beiden Teilmengen enthalten sind.

In [4]:
df = (raw
      .select(col("headline").alias("text"),
              col("is_sarcastic").cast("int").alias("label"))
      .na.drop(subset=["text","label"]))

train, test = df.randomSplit([0.8, 0.2], seed=42)
df.count(), train.count(), test.count()

(28619, 22932, 5687)

## Merkmalsextraktion und Modelltraining

In diesem Schritt wird eine vollständige Spark-ML-Pipeline aufgebaut und auf den Trainingsdaten trainiert.

Zunächst wird der Text mit einem `RegexTokenizer` in einzelne Wörter zerlegt. Anschließend entfernt `StopWordsRemover` häufige englische Stoppwörter. Mit `NGram` werden zusätzlich Bi-Gramme erzeugt, also Wortpaare, die lokale Wortzusammenhänge erfassen.

Danach werden die Uni-Gramme und Bi-Gramme jeweils mit `CountVectorizer` in numerische Häufigkeitsvektoren umgewandelt. Diese beiden Vektoren werden mit `VectorAssembler` zu einem gemeinsamen Merkmalsvektor kombiniert und anschließend mit `IDF` gewichtet, sodass seltene und damit oft aussagekräftigere Terme stärker berücksichtigt werden.

Als Klassifikationsmodell wird eine logistische Regression verwendet. Alle Verarbeitungsschritte werden in einer `Pipeline` zusammengefasst, sodass Vorverarbeitung und Training als ein zusammenhängender Workflow ausgeführt werden können. Mit `fit(train)` wird das Modell schließlich auf den Trainingsdaten gelernt.

In [5]:
tok = RegexTokenizer(inputCol="text", outputCol="tokens", pattern="\\W+", minTokenLength=2)
sw  = StopWordsRemover(inputCol="tokens", outputCol="tokens_clean")
bi  = NGram(n=2, inputCol="tokens_clean", outputCol="bigrams")

cv_uni = CountVectorizer(inputCol="tokens_clean", outputCol="tf_uni", minDF=5)
cv_bi  = CountVectorizer(inputCol="bigrams",      outputCol="tf_bi",  minDF=5)

stack = VectorAssembler(inputCols=["tf_uni","tf_bi"], outputCol="tf")
idf   = IDF(inputCol="tf", outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="label",
                        maxIter=50, regParam=0.1, elasticNetParam=0.0)

pipe = Pipeline(stages=[tok, sw, bi, cv_uni, cv_bi, stack, idf, lr])
model = pipe.fit(train)


### Inferenz und Sichtkontrolle der Features

Nachdem das Modell trainiert wurde, wird es auf die Testdaten angewendet.  
Mit `model.transform(test)` werden für jede Zeile die zuvor definierten Verarbeitungsschritte der Pipeline ausgeführt und die entsprechenden Feature-Vektoren erzeugt.

Die anschließende Ausgabe zeigt das tatsächliche Label (`label`) sowie den berechneten Feature-Vektor (`features`).  

In [6]:
pred = model.transform(test)
pred.select("label", "features").show(truncate=False)

+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|label|features                                                                                                                                                                                                                                                                                                         |
+-----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1    |(7260,[2,6,11,926,2139,6833],[2.938656228644528,3.9

### Evaluation

Nach der Anwendung des Modells auf die Testdaten werden verschiedene Evaluationsmetriken berechnet. Die Vorhersagen werden zunächst mit `cache()` im Speicher gehalten, damit sie für mehrere Auswertungen effizient wiederverwendet werden können.

Zur Bewertung der Klassifikationsleistung werden mehrere Kennzahlen verwendet:

- **ROC-AUC**: Misst die Trennschärfe des Modells über verschiedene Schwellenwerte hinweg.
- **PR-AUC**: Besonders aussagekräftig bei unausgeglichenen Klassenverteilungen.
- **F1-Score**: Harmonisches Mittel aus Präzision und Recall.
- **Accuracy**: Anteil korrekt klassifizierter Beispiele.

Zusätzlich wird eine Konfusionsmatrix ausgegeben. Sie zeigt, wie viele Beispiele jeder tatsächlichen Klasse (`label`) als welche Klasse (`prediction`) vorhergesagt wurden und hilft dabei, typische Fehlklassifikationen zu erkennen.

In [7]:
pred = model.transform(test).cache()

roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(pred)
pr  = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderPR").evaluate(pred)
f1  = MulticlassClassificationEvaluator(labelCol="label", metricName="f1").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy").evaluate(pred)
print(f"ROC-AUC: {roc:.3f} | PR-AUC: {pr:.3f} | F1: {f1:.3f} | Acc: {acc:.3f}")

print("\nKonfusionsmatrix:")
(pred.groupBy("label","prediction").count()
     .orderBy("label","prediction").show())


ROC-AUC: 0.876 | PR-AUC: 0.874 | F1: 0.795 | Acc: 0.795

Konfusionsmatrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 2497|
|    0|       1.0|  471|
|    1|       0.0|  692|
|    1|       1.0| 2027|
+-----+----------+-----+



### Modellinterpretation: wichtige Wörter und Bi-gramme

In diesem Schritt werden die gelernten Gewichte des logistischen Regressionsmodells analysiert, um zu verstehen, welche Wörter oder Wortpaare besonders stark zur Sarkasmus-Erkennung beitragen.

Zunächst werden aus der trainierten Pipeline die relevanten Modelle extrahiert: die beiden `CountVectorizer`-Modelle für Uni-gramme und Bi-gramme sowie das trainierte Regressionsmodell. Aus den `CountVectorizer`-Modellen wird das jeweilige Vokabular ausgelesen, während das Regressionsmodell die Koeffizienten der gelernten Feature-Gewichte liefert.

Da die Feature-Vektoren aus Uni-grammen und Bi-grammen kombiniert wurden, werden die Koeffizienten entsprechend aufgeteilt und mit den jeweiligen Tokens verknüpft.

Anschließend werden drei Listen ausgegeben:

- **Top-20 Bi-gramme** mit den höchsten positiven Gewichten – sie sind starke Hinweise auf Sarkasmus.
- **Top-20 Uni-gramme** mit positiven Gewichten – einzelne Wörter, die häufig mit sarkastischen Überschriften verbunden sind.
- **Top-20 negative Gewichte** – Wörter oder Wortpaare, die eher gegen Sarkasmus sprechen.

Diese Analyse ermöglicht eine einfache **Interpretation des Modells**, indem sichtbar wird, welche sprachlichen Muster für die Klassifikation besonders relevant sind.

In [ ]:
# Pipeline-Stages extrahieren
stages = model.stages
cv_uni_model = stages[3]
cv_bi_model  = stages[4]
lr_model     = stages[7]

v_uni = cv_uni_model.vocabulary
v_bi  = cv_bi_model.vocabulary


coef = lr_model.coefficients.toArray()  # entspricht Reihenfolge der TF-IDF-Features

k_uni = len(v_uni)
pairs_uni = list(zip(v_uni, coef[:k_uni]))
pairs_bi  = list(zip(v_bi,  coef[k_uni:]))

def topk(pairs, k=20):   return sorted(pairs, key=lambda x: x[1], reverse=True)[:k]
def bottomk(pairs, k=20):return sorted(pairs, key=lambda x: x[1])[:k]

print("\nTop-20 Bi-gramme (Sarkasmus-Hinweise):")
for w,c in topk(pairs_bi, 20): print(f"{w:35s} {c:+.3f}")

print("\nTop-20 Unigramme (Sarkasmus-Hinweise):")
for w,c in topk(pairs_uni, 20): print(f"{w:35s} {c:+.3f}")

print("\nTop-20 Anti-Sarkasmus-Anker (neg. Gewichte):")
for w,c in bottomk(pairs_uni + pairs_bi, 20): print(f"{w:35s} {c:+.3f}")



Top-20 Bi-gramme (Sarkasmus-Hinweise):
american people                     +0.238
healthcare bill                     +0.233
federal government                  +0.219
trump boys                          +0.215
dead body                           +0.208
talk show                           +0.204
footage reveals                     +0.194
number one                          +0.189
releases new                        +0.188
30 million                          +0.188
small town                          +0.184
one time                            +0.183
says man                            +0.178
claims life                         +0.177
researchers find                    +0.174
capitol building                    +0.174
debuts new                          +0.174
state dinner                        +0.168
come forward                        +0.167
pregnant woman                      +0.165

Top-20 Unigramme (Sarkasmus-Hinweise):
nation                              +0.299
passage          

## Beispielvorhersagen mit neuen Texten

Zum Abschluss wird das trainierte Modell auf einige neue, selbst definierte Beispielsätze angewendet. Dazu wird zunächst ein DataFrame mit den Testtexten erstellt und anschließend durch die Pipeline transformiert.

Aus dem Ergebnis wird die Wahrscheinlichkeit für die Klasse „sarkastisch“ aus dem `probability`-Vektor extrahiert. Zusätzlich wird die vom Modell vorhergesagte Klasse (`prediction`) angezeigt.

Die Ausgabe zeigt somit für jeden Beispielsatz:
- den ursprünglichen Text,
- die berechnete Wahrscheinlichkeit für Sarkasmus (`p_sarcasm`),
- sowie die vorhergesagte Klasse.

Die Ergebnisse werden nach der Sarkasmus-Wahrscheinlichkeit sortiert, sodass die am stärksten als sarkastisch eingeschätzten Texte zuerst erscheinen.

In [8]:
from pyspark.sql import Row

examples = [
    "Local man heroically volunteers to finish leftover pizza",
    "Government announces new plan to reduce taxes next quarter",
    "Experts confirm Mondays are indeed the worst"
]

ex_df = spark.createDataFrame([Row(text=t) for t in examples])
out = model.transform(ex_df)

to_prob1 = F.udf(lambda v: float(v[1]), "double")
(out.select("text", to_prob1("probability").alias("p_sarcasm"),
            F.col("prediction").cast("int").alias("pred"))
   .orderBy(F.desc("p_sarcasm"))
   .show(truncate=False))


+----------------------------------------------------------+------------------+----+
|text                                                      |p_sarcasm         |pred|
+----------------------------------------------------------+------------------+----+
|Government announces new plan to reduce taxes next quarter|0.9349447141547973|1   |
|Local man heroically volunteers to finish leftover pizza  |0.9343669974600858|1   |
|Experts confirm Mondays are indeed the worst              |0.5098573787942909|1   |
+----------------------------------------------------------+------------------+----+

